In [1]:
import os,sys
# Set the path to the parent directory manually
parent_dir = os.path.abspath("../..")
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime as dt, timedelta
from datetime import datetime
import glob
from netCDF4 import Dataset
from util.wrf_process import (calc_derive, object_tracking, read_and_write, to_polar)
from util.ml_framework import (cnn, vae)
from wrf import (to_np, getvar, smooth2d, get_cartopy, cartopy_xlim, interplevel, to_np,
                 cartopy_ylim, latlon_coords, interplevel, ll_to_xy)
import gc,pickle
from tqdm import tqdm
import xarray as xr
from natsort import natsorted
from util.ml_preprocess import data_preproc
import proplot as plot
from proc_vars import get_precip_class, area_stats
from metpy.units import units

/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Track path
track_memb03 = xr.open_dataset('/glade/derecho/scratch/ihtam/temp/memb07/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/temp/memb07/wrfout_d02_2013-11-0*"))[1:30]
mdd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

plevs = np.array([1000, 975, 950, 925, 900, 875, 850, 800, 750, 700, 650, 600, 550, 500, 450, 400, 350, 300, 250, 200, 150, 100])
#plevs_full = np.array([1000, 975, 950, 925, 900, 875, 850, 800, 750, 700, 650, 600, 550, 500, 450, 400, 350, 300, 250, 200, 150, 100])

In [4]:
def calc_surfvert(ncfile):
    return np.sqrt(getvar(ncfile,'ua')[0,...]**2+getvar(ncfile,'va')[0,...]**2)

def layer_mean(field_p, plev, pmin, pmax):
    # field_p: (plev,y,x), pmin/pmax in hPa (e.g. 850, 950)
    mask = (plevs <= pmax) & (plevs >= pmin)
    return np.nanmean(field_p[mask, :, :], axis=0)  # 2D

def calc_wa_mean(ncfile,plevs,pmin,pmax):
    # Read in wa
    ctrl_wa = getvar(ncfile,'wa')
    # Read in pressure
    ctrl_p = getvar(ncfile,'pressure')
    # Interpolate AVOR to pressure level
    wa_p = interplevel(ctrl_wa, ctrl_p, plevs)
    # Using layer mean vorticity
    return layer_mean(wa_p, plevs, pmin=pmin, pmax=pmax)
    
def calc_avor_p(ncfile,plevs,pmin,pmax):
    # Read in AVOR
    ctrl_avo = getvar(ncfile,'avo')
    # Read in pressure
    ctrl_p = getvar(ncfile,'pressure')
    # Interpolate AVOR to pressure level
    avor_p = interplevel(ctrl_avo, ctrl_p, plevs)
    # Using layer mean vorticity
    return layer_mean(avor_p, plevs, pmin=pmin, pmax=pmax)

def calc_pvo_p(ncfile,plevs,pmin,pmax):
    # Read in AVOR
    ctrl_avo = getvar(ncfile,'pvo')
    # Read in pressure
    ctrl_p = getvar(ncfile,'pressure')
    # Interpolate AVOR to pressure level
    avor_p = interplevel(ctrl_avo, ctrl_p, plevs)
    # Using layer mean vorticity
    return layer_mean(avor_p, plevs, pmin=pmin, pmax=pmax)

def calc_rthratlw_p(ncfile,plevs,pmin,pmax):
    # Read in AVOR
    ctrl_avo = getvar(ncfile,'RTHRATLW')
    # Read in pressure
    ctrl_p = getvar(ncfile,'pressure')
    # Interpolate AVOR to pressure level
    avor_p = interplevel(ctrl_avo, ctrl_p, plevs)
    # Using layer mean vorticity
    return layer_mean(avor_p, plevs, pmin=pmin, pmax=pmax)

def calc_qv_p(ncfile,plevs,pmin,pmax):
    # Read in QVAPOR
    ctrl_qv = getvar(ncfile,'QVAPOR')
    # Read in pressure
    ctrl_p = getvar(ncfile,'pressure')
    # Interpolate QVAPOR to pressure level
    avor_qv = interplevel(ctrl_qv, ctrl_p, plevs)
    # Using layer mean vorticity
    return layer_mean(avor_qv, plevs, pmin=pmin, pmax=pmax)

def calc_conv(ncfile, plevs, pmin, pmax, timeidx=None):
    import numpy as np
    from metpy.calc import divergence
    from metpy.units import units
    from wrf import getvar, interplevel, to_np

    # --- Read winds + pressure ---
    # If you want a single time, pass timeidx=0 (or any int). If None, keep all times.
    ua = getvar(ncfile, "ua")   # (Time,z,y,x) or (z,y,x)
    va = getvar(ncfile, "va")
    p  = getvar(ncfile, "pressure")

    # --- Interpolate to pressure levels ---
    ua_p = interplevel(ua, p, plevs)  # (Time,plev,y,x) or (plev,y,x)
    va_p = interplevel(va, p, plevs)

    # --- Grid spacing ---
    dx = float(ncfile.DX) * units.m
    dy = float(ncfile.DY) * units.m

    # --- Divergence over the horizontal dims (last two dims) for ALL leading dims at once ---
    div_p = divergence(
        np.asarray(ua_p) * units("m/s"),
        np.asarray(va_p) * units("m/s"),
        dx=dx,
        dy=dy,
        x_dim=-1, y_dim=-2
    )  # units 1/s, shape matches ua_p: (Time,plev,y,x) or (plev,y,x)

    div_p = div_p.to("1/s").magnitude

    # --- Layer-mean convergence: -div averaged over pmin..pmax ---
    # Assuming your layer_mean expects (plev,y,x) OR (Time,plev,y,x) and uses plevs as the vertical coord:
    conv_ll_2d = -layer_mean(div_p, plevs, pmin=pmin, pmax=pmax)

    return conv_ll_2d
    
def calc_precipclass(ncfile,ix,iy,ra):
    # Get Precip Class
    ctrl_pclass = get_precip_class(ncfile)[int(iy)-ra:int(iy)+ra,int(ix)-ra:int(ix)+ra]
    # Derive axisymmetry
    ctrl_pclasses = {'conv':area_stats(np.sum(ctrl_pclass==1), 4*ra*ra), 'strat':area_stats(np.sum(ctrl_pclass==4), 4*ra*ra),
                     'cong':area_stats(np.sum(ctrl_pclass==2),4*ra*ra), 'shal':area_stats(np.sum(ctrl_pclass==3),4*ra*ra),
                     'anvil':area_stats(np.sum(ctrl_pclass==5),4*ra*ra)}
    return ctrl_pclasses

In [6]:
def proc_onevar_loop(fileslist,track,dr,TYPE,plevs):
    varouts = []
    for i in tqdm(range(len(fileslist))):
        nc_ctrl = Dataset(fileslist[i])
        # Target location
        target_lat = track['clat'][12+1+i]
        target_lon = track['clon'][12+1+i]
        # Compute track indices
        ix, iy = ll_to_xy(nc_ctrl, target_lat, target_lon, timeidx=0) #x, y
        if TYPE=='slp':
            ctrl_slp = getvar(nc_ctrl,'slp')
            try:
                varouts.append(float(ctrl_slp[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].min()))
            except:
                varouts.append(np.nan)
            del ctrl_slp
            gc.collect()
        elif TYPE=='lpvort':
            lvort = calc_pvo_p(nc_ctrl, plevs, 850, 1000)
            varouts.append(float(lvort[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
            del lvort
            gc.collect()
        elif TYPE=='mpvort':
            mvort = calc_pvo_p(nc_ctrl, plevs, 500, 700)
            varouts.append(float(mvort[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
            del mvort
            gc.collect()
        elif TYPE=='lvort':
            lvort = calc_avor_p(nc_ctrl, plevs, 850, 1000)
            varouts.append(float(lvort[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
            del lvort
            gc.collect()
        elif TYPE=='mvort':
            mvort = calc_avor_p(nc_ctrl, plevs, 500, 700)
            varouts.append(float(mvort[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
            del mvort
            gc.collect()
        elif TYPE=='pclasses':
            varouts.append(calc_precipclass(fileslist[i],ix,iy,dr))
        elif TYPE=='lvapor':
            lqv = calc_qv_p(nc_ctrl, plevs, 850, 1000)
            varouts.append(float(lqv[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
        elif TYPE=='mvapor':
            mqv = calc_qv_p(nc_ctrl, plevs, 500, 700)
            varouts.append(float(mqv[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
        elif TYPE=='lconv':
            lconv = calc_conv(nc_ctrl, plevs, 850, 1000)
            varouts.append(float(lconv[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
        elif TYPE=='mwa_mean':
            mwamean = calc_wa_mean(nc_ctrl, plevs, 500, 850)
            varouts.append(float(mwamean[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
        elif TYPE=='ulw':
            ulw = calc_rthratlw_p(nc_ctrl, plevs, 200, 400)
            varouts.append(float(ulw[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
        elif TYPE=='mlw':
            mlw = calc_rthratlw_p(nc_ctrl, plevs, 500, 700)
            varouts.append(float(mlw[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))            
    return varouts

In [9]:
# Track path
track_memb07 = xr.open_dataset('/glade/derecho/scratch/ihtam/temp/memb07/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/temp/memb07/wrfout_d02_2013-11-0*"))[1:30]
mdd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

# Calculate Low-level Vorticity
ctrl_lpvort = proc_onevar_loop(ctrl_files,track_memb07,100,'lpvort',plevs)
dd_p2f_lpvort = proc_onevar_loop(dd_p2f_files,track_memb07,100,'lpvort',plevs)
mdd_p2f_lpvort = proc_onevar_loop(mdd_p2f_files,track_memb07,100,'lpvort',plevs)
# Calculate mid-level Vorticity
ctrl_mpvort = proc_onevar_loop(ctrl_files,track_memb07,100,'mpvort',plevs)
dd_p2f_mpvort = proc_onevar_loop(dd_p2f_files,track_memb07,100,'mpvort',plevs)
mdd_p2f_mpvort = proc_onevar_loop(mdd_p2f_files,track_memb07,100,'mpvort',plevs)

lpvortdict = {'ctrl':ctrl_lpvort,'mdd_p2f':mdd_p2f_lpvort,'dd_p2f':dd_p2f_lpvort}
with open(f"./store/haiyansen_memb07_lpvort.pkl", "wb") as f:
    pickle.dump(lpvortdict, f)

mpvortdict = {'ctrl':ctrl_mpvort,'mdd_p2f':mdd_p2f_mpvort,'dd_p2f':dd_p2f_mpvort}
with open(f"./store/haiyansen_memb07_mpvort.pkl", "wb") as f:
    pickle.dump(mpvortdict, f)

100%|██████████| 29/29 [04:27<00:00,  9.24s/it]


## Vorticity£

In [8]:
# Track path
track_memb07 = xr.open_dataset('/glade/derecho/scratch/ihtam/temp/memb07/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/temp/memb07/wrfout_d02_2013-11-0*"))[1:30]
mdd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

# Calculate Low-level Vorticity
ctrl_lvort = proc_onevar_loop(ctrl_files,track_memb07,100,'lvort',plevs)
dd_p2f_lvort = proc_onevar_loop(dd_p2f_files,track_memb07,100,'lvort',plevs)
mdd_p2f_lvort = proc_onevar_loop(mdd_p2f_files,track_memb07,100,'lvort',plevs)
# Calculate mid-level Vorticity
ctrl_mvort = proc_onevar_loop(ctrl_files,track_memb07,100,'mvort',plevs)
dd_p2f_mvort = proc_onevar_loop(dd_p2f_files,track_memb07,100,'mvort',plevs)
mdd_p2f_mvort = proc_onevar_loop(mdd_p2f_files,track_memb07,100,'mvort',plevs)

lvortdict = {'ctrl':ctrl_lvort,'mdd_p2f':mdd_p2f_lvort,'dd_p2f':dd_p2f_lvort}
with open(f"./store/haiyansen_memb07_lvort.pkl", "wb") as f:
    pickle.dump(lvortdict, f)

mvortdict = {'ctrl':ctrl_mvort,'mdd_p2f':mdd_p2f_mvort,'dd_p2f':dd_p2f_mvort}
with open(f"./store/haiyansen_memb07_mvort.pkl", "wb") as f:
    pickle.dump(mvortdict, f)

  0%|          | 0/29 [00:07<?, ?it/s]

KeyboardInterrupt



In [5]:
track_memb03 = xr.open_dataset('/glade/work/ihtam/storage/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/ctrl/wrfout_d02_2013-11-0*"))[:29]
miaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
aiaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/add_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
mdd_p2f_files =sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

expname = ['ctrl','miaxi2x','aiaxi2x','mdd_p2f','dd_p2f']
files = [ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files]
del ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files
gc.collect()

lvortdict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'lvort',plevs) for i in range(5)}
mvortdict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'mvort',plevs) for i in range(5)}

with open(f"./store/haiyansen_memb03_lvort.pkl", "wb") as f:
    pickle.dump(lvortdict_memb03, f)

with open(f"./store/haiyansen_memb03_mvort.pkl", "wb") as f:
    pickle.dump(mvortdict_memb03, f)

100%|██████████| 29/29 [02:58<00:00,  6.15s/it]


In [6]:
oldensm_lvort = {}
oldensm_mvort = {}
for i in ['01','02','03','04','05','06','07','08','09','10']:
    track_memb03 = xr.open_dataset(f'/glade/derecho/scratch/ihtam/temp/memb{i}/track_avor_850-600.nc')
    ctrl_files = sorted(glob.glob(f"/glade/derecho/scratch/ihtam/temp/memb{i}/wrfout_d02_2013-11-0*"))[1:30]
    oldensm_lvort[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'lvort',plevs)
    oldensm_mvort[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'mvort',plevs)
    
with open(f"./store/haiyan_oldensm_lvort.pkl", "wb") as f:
    pickle.dump(oldensm_lvort, f)
with open(f"./store/haiyan_oldensm_mvort.pkl", "wb") as f:
    pickle.dump(oldensm_mvort, f)

 62%|██████▏   | 18/29 [01:59<01:13,  6.70s/it]/glade/derecho/scratch/ihtam/tmp/ipykernel_24098/2669730760.py:20: RuntimeWarning: Mean of empty slice.
  varouts.append(float(lvort[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
 62%|██████▏   | 18/29 [02:01<01:13,  6.72s/it]/glade/derecho/scratch/ihtam/tmp/ipykernel_24098/2669730760.py:25: RuntimeWarning: Mean of empty slice.
  varouts.append(float(mvort[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
 69%|██████▉   | 20/29 [02:10<00:59,  6.65s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid

## Moisture

In [11]:
# Track path
track_memb07 = xr.open_dataset('/glade/derecho/scratch/ihtam/temp/memb07/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/temp/memb07/wrfout_d02_2013-11-0*"))[1:30]
mdd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
# Calculate Low-level Vapor
ctrl_lvapor = proc_onevar_loop(ctrl_files,track_memb07,100,'lvapor',plevs)
dd_p2f_lvapor = proc_onevar_loop(dd_p2f_files,track_memb07,100,'lvapor',plevs)
mdd_p2f_lvapor = proc_onevar_loop(mdd_p2f_files,track_memb07,100,'lvapor',plevs)
# Calculate mid-level Vapor
ctrl_mvapor = proc_onevar_loop(ctrl_files,track_memb07,100,'mvapor',plevs)
dd_p2f_mvapor = proc_onevar_loop(dd_p2f_files,track_memb07,100,'mvapor',plevs)
mdd_p2f_mvapor = proc_onevar_loop(mdd_p2f_files,track_memb07,100,'mvapor',plevs)

lvapordict = {'ctrl':ctrl_lvapor,'mdd_p2f':mdd_p2f_lvapor,'dd_p2f':dd_p2f_lvapor}
with open(f"./store/haiyansen_memb07_lvapor.pkl", "wb") as f:
    pickle.dump(lvapordict, f)
mvapordict = {'ctrl':ctrl_mvapor,'mdd_p2f':mdd_p2f_mvapor,'dd_p2f':dd_p2f_mvapor}
with open(f"./store/haiyansen_memb07_mvapor.pkl", "wb") as f:
    pickle.dump(mvapordict, f)

100%|██████████| 29/29 [01:40<00:00,  3.48s/it]


In [6]:
track_memb03 = xr.open_dataset('/glade/work/ihtam/storage/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/ctrl/wrfout_d02_2013-11-0*"))[:29]
miaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
aiaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/add_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
mdd_p2f_files =sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

expname = ['ctrl','miaxi2x','aiaxi2x','mdd_p2f','dd_p2f']
files = [ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files]
del ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files
gc.collect()
lvapordict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'lvapor',plevs) for i in range(5)}
mvapordict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'mvapor',plevs) for i in range(5)}

with open(f"./store/haiyansen_memb03_lvapor.pkl", "wb") as f:
    pickle.dump(lvapordict_memb03, f)

with open(f"./store/haiyansen_memb03_mvapor.pkl", "wb") as f:
    pickle.dump(mvapordict_memb03, f)

100%|██████████| 29/29 [01:30<00:00,  3.13s/it]


In [7]:
oldensm_lvapor = {}
oldensm_mvapor = {}
for i in ['01','02','03','04','05','06','07','08','09','10']:
    track_memb03 = xr.open_dataset(f'/glade/derecho/scratch/ihtam/temp/memb{i}/track_avor_850-600.nc')
    ctrl_files = sorted(glob.glob(f"/glade/derecho/scratch/ihtam/temp/memb{i}/wrfout_d02_2013-11-0*"))[1:30]
    oldensm_lvapor[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'lvapor',plevs)
    oldensm_mvapor[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'mvapor',plevs)
    
with open(f"./store/haiyan_oldensm_lvapor.pkl", "wb") as f:
    pickle.dump(oldensm_lvapor, f)
with open(f"./store/haiyan_oldensm_mvapor.pkl", "wb") as f:
    pickle.dump(oldensm_mvapor, f)

 62%|██████▏   | 18/29 [01:08<00:41,  3.79s/it]/glade/derecho/scratch/ihtam/tmp/ipykernel_6117/2085755570.py:32: RuntimeWarning: Mean of empty slice.
  varouts.append(float(lqv[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
 62%|██████▏   | 18/29 [01:09<00:39,  3.56s/it]/glade/derecho/scratch/ihtam/tmp/ipykernel_6117/2085755570.py:35: RuntimeWarning: Mean of empty slice.
  varouts.append(float(mqv[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
 69%|██████▉   | 20/29 [01:27<00:41,  4.59s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value

## Stratiform

In [23]:
# Track path
track_memb03 = xr.open_dataset('/glade/derecho/scratch/ihtam/temp/memb07/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/temp/memb07/wrfout_d02_2013-11-0*"))[1:30]
mdd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
# Calculate Precipitation classes
ctrl_pclasses = proc_onevar_loop(ctrl_files,track_memb03,100,'pclasses')
dd_p2f_pclasses = proc_onevar_loop(dd_p2f_files,track_memb03,100,'pclasses')
mdd_p2f_pclasses = proc_onevar_loop(mdd_p2f_files,track_memb03,100,'pclasses')

pclassesdict = {'ctrl':ctrl_pclasses,'mdd_p2f':mdd_p2f_pclasses,'dd_p2f':dd_p2f_pclasses}
with open(f"./store/haiyansen_memb07_pclasses.pkl", "wb") as f:
    pickle.dump(pclassesdict, f)


100%|██████████| 29/29 [01:48<00:00,  3.75s/it]


In [8]:
track_memb03 = xr.open_dataset('/glade/work/ihtam/storage/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/ctrl/wrfout_d02_2013-11-0*"))[:29]
miaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
aiaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/add_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
mdd_p2f_files =sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

expname = ['ctrl','miaxi2x','aiaxi2x','mdd_p2f','dd_p2f']
files = [ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files]
del ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files
gc.collect()

pclassesdict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'pclasses',plevs) for i in range(5)}

with open(f"./store/haiyansen_memb03_pclasses.pkl", "wb") as f:
    pickle.dump(pclassesdict_memb03, f)

100%|██████████| 29/29 [01:49<00:00,  3.77s/it]


In [6]:
oldensm_pclasses = {}
oldensm_pclasses = {}
for i in ['01','02','03','04','05','06','07','08','09','10']:
    track_memb03 = xr.open_dataset(f'/glade/derecho/scratch/ihtam/temp/memb{i}/track_avor_850-600.nc')
    ctrl_files = sorted(glob.glob(f"/glade/derecho/scratch/ihtam/temp/memb{i}/wrfout_d02_2013-11-0*"))[1:30]
    oldensm_pclasses[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'pclasses',plevs)
    
with open(f"./store/haiyan_oldensm_pclasses.pkl", "wb") as f:
    pickle.dump(oldensm_pclasses, f)

 69%|██████▉   | 20/29 [01:14<00:33,  3.70s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
 72%|███████▏  | 21/29 [01:17<00:29,  3.71s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
 76%|███████▌  | 22/29 [01:21<00:25,  3.70s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
 79%|███████▉  | 23/29 [01:25<00:22,  3.69s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
 83%|████████▎ | 24/29 [01:29<00:18,  3.73s/it]/glade/work/ihtam/miniconda3/envs/myenv/l

## Convergence

In [12]:
# Calculate Precipitation classes
track_memb07 = xr.open_dataset('/glade/derecho/scratch/ihtam/temp/memb07/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/temp/memb07/wrfout_d02_2013-11-0*"))[1:30]
mdd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

ctrl_lconv = proc_onevar_loop(ctrl_files,track_memb07,100,'lconv',plevs)
dd_p2f_lconv = proc_onevar_loop(dd_p2f_files,track_memb07,100,'lconv',plevs)
mdd_p2f_lconv = proc_onevar_loop(mdd_p2f_files,track_memb07,100,'lconv',plevs)

lconvdict = {'ctrl':ctrl_lconv,'mdd_p2f':mdd_p2f_lconv,'dd_p2f':dd_p2f_lconv}
with open(f"./store/haiyansen_memb07_lconv.pkl", "wb") as f:
    pickle.dump(lconvdict, f)

100%|██████████| 29/29 [04:15<00:00,  8.81s/it]


In [25]:
track_memb03 = xr.open_dataset('/glade/work/ihtam/storage/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/ctrl/wrfout_d02_2013-11-0*"))[:29]
miaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
aiaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/add_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
mdd_p2f_files =sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

expname = ['ctrl','miaxi2x','aiaxi2x','mdd_p2f','dd_p2f']
files = [ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files]
del ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files
gc.collect()

lconvdict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'lconv',plevs) for i in range(5)}
with open(f"./store/haiyansen_memb03_lconv.pkl", "wb") as f:
    pickle.dump(lconvdict_memb03, f)

100%|██████████| 29/29 [03:51<00:00,  8.00s/it]


In [10]:
oldensm_lconv = {}
for i in ['01','02','03','04','05','06','07','08','09','10']:
    track_memb03 = xr.open_dataset(f'/glade/derecho/scratch/ihtam/temp/memb{i}/track_avor_850-600.nc')
    ctrl_files = sorted(glob.glob(f"/glade/derecho/scratch/ihtam/temp/memb{i}/wrfout_d02_2013-11-0*"))[1:30]
    oldensm_lconv[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'lconv',plevs)
    
with open(f"./store/haiyan_oldensm_lconv.pkl", "wb") as f:
    pickle.dump(oldensm_lconv, f)

 62%|██████▏   | 18/29 [02:30<01:31,  8.36s/it]/glade/derecho/scratch/ihtam/tmp/ipykernel_77043/2119895538.py:35: RuntimeWarning: Mean of empty slice.
  varouts.append(float(lconv[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
 69%|██████▉   | 20/29 [02:55<01:20,  8.91s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
/glade/derecho/scratch/ihtam/tmp/ipykernel_77043/2119895538.py:35: RuntimeWarning: Mean of empty slice.
  varouts.append(float(lconv[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  re

## MSLP

In [13]:
track_memb07 = xr.open_dataset('/glade/derecho/scratch/ihtam/temp/memb07/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/temp/memb07/wrfout_d02_2013-11-0*"))[1:30]
mdd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

ctrl_slp= proc_onevar_loop(ctrl_files,track_memb07,100,'slp')
dd_p2f_slp= proc_onevar_loop(dd_p2f_files,track_memb07,100,'slp')
mdd_p2f_slp= proc_onevar_loop(mdd_p2f_files,track_memb07,100,'slp')

slpdict = {'ctrl':ctrl_slp,'mdd_p2f':mdd_p2f_slp,'dd_p2f':dd_p2f_slp}
with open(f"./store/haiyansen_memb07_slp.pkl", "wb") as f:
    pickle.dump(slpdict, f)

100%|██████████| 29/29 [02:14<00:00,  4.64s/it]


In [13]:
track_memb03 = xr.open_dataset('/glade/work/ihtam/storage/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/ctrl/wrfout_d02_2013-11-0*"))[:29]
miaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
aiaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/add_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
mdd_p2f_files =sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

expname = ['ctrl','miaxi2x','aiaxi2x','mdd_p2f','dd_p2f']
files = [ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files]
del ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files
gc.collect()

slpdict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'slp',plevs) for i in range(5)}
with open(f"./store/haiyansen_memb03_slp.pkl", "wb") as f:
    pickle.dump(slpdict_memb03, f)

100%|██████████| 29/29 [02:30<00:00,  5.20s/it]


In [5]:
oldensm_slp = {}
for i in ['01','02','03','04','05','06','07','08','09','10']:
    track_memb03 = xr.open_dataset(f'/glade/derecho/scratch/ihtam/temp/memb{i}/track_avor_850-600.nc')
    ctrl_files = sorted(glob.glob(f"/glade/derecho/scratch/ihtam/temp/memb{i}/wrfout_d02_2013-11-0*"))[1:30]
    oldensm_slp[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'slp',plevs)
    
with open(f"./store/haiyan_oldensm_slp.pkl", "wb") as f:
    pickle.dump(oldensm_slp, f)

 69%|██████▉   | 20/29 [01:39<00:46,  5.12s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
 72%|███████▏  | 21/29 [01:44<00:41,  5.14s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
 76%|███████▌  | 22/29 [01:49<00:35,  5.13s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
 79%|███████▉  | 23/29 [01:54<00:30,  5.11s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
 83%|████████▎ | 24/29 [01:59<00:25,  5.10s/it]/glade/work/ihtam/miniconda3/envs/myenv/l

## wa

In [18]:
track_memb07 = xr.open_dataset('/glade/derecho/scratch/ihtam/temp/memb07/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/temp/memb07/wrfout_d02_2013-11-0*"))[1:30]
mdd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

ctrl_mwa_mean= proc_onevar_loop(ctrl_files,track_memb07,100,'mwa_mean',plevs)
dd_p2f_mwa_mean= proc_onevar_loop(dd_p2f_files,track_memb07,100,'mwa_mean',plevs)
mdd_p2f_mwa_mean= proc_onevar_loop(mdd_p2f_files,track_memb07,100,'mwa_mean',plevs)

mwa_meandict = {'ctrl':ctrl_mwa_mean,'mdd_p2f':mdd_p2f_mwa_mean,'dd_p2f':dd_p2f_mwa_mean}
with open(f"./store/haiyansen_memb07_mwa_mean.pkl", "wb") as f:
    pickle.dump(mwa_meandict, f)

100%|██████████| 29/29 [02:02<00:00,  4.21s/it]


In [19]:
track_memb03 = xr.open_dataset('/glade/work/ihtam/storage/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/ctrl/wrfout_d02_2013-11-0*"))[:29]
miaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
aiaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/add_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
mdd_p2f_files =sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

expname = ['ctrl','miaxi2x','aiaxi2x','mdd_p2f','dd_p2f']
files = [ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files]
del ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files
gc.collect()

mwa_meandict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'mwa_mean',plevs) for i in range(5)}
with open(f"./store/haiyansen_memb03_mwa_mean.pkl", "wb") as f:
    pickle.dump(mwa_meandict_memb03, f)

100%|██████████| 29/29 [02:05<00:00,  4.31s/it]


In [20]:
oldensm_mwa_mean = {}
for i in ['01','02','03','04','05','06','07','08','09','10']:
    track_memb03 = xr.open_dataset(f'/glade/derecho/scratch/ihtam/temp/memb{i}/track_avor_850-600.nc')
    ctrl_files = sorted(glob.glob(f"/glade/derecho/scratch/ihtam/temp/memb{i}/wrfout_d02_2013-11-0*"))[1:30]
    oldensm_mwa_mean[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'mwa_mean',plevs)
    
with open(f"./store/haiyan_oldensm_mwa_mean.pkl", "wb") as f:
    pickle.dump(oldensm_mwa_mean, f)

 62%|██████▏   | 18/29 [01:16<00:47,  4.32s/it]/glade/derecho/scratch/ihtam/tmp/ipykernel_6117/162560278.py:41: RuntimeWarning: Mean of empty slice.
  varouts.append(float(mwamean[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
 69%|██████▉   | 20/29 [01:26<00:37,  4.18s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid value encountered in cast
  result = np.rint(result).astype(int)
/glade/derecho/scratch/ihtam/tmp/ipykernel_6117/162560278.py:41: RuntimeWarning: Mean of empty slice.
  varouts.append(float(mwamean[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.

## RTHRATLW

In [30]:
track_memb07 = xr.open_dataset('/glade/derecho/scratch/ihtam/temp/memb07/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/temp/memb07/wrfout_d02_2013-11-0*"))[1:30]
mdd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/memb07_add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

ctrl_ulw= proc_onevar_loop(ctrl_files,track_memb07,100,'ulw',plevs)
dd_p2f_ulw= proc_onevar_loop(dd_p2f_files,track_memb07,100,'ulw',plevs)
mdd_p2f_ulw= proc_onevar_loop(mdd_p2f_files,track_memb07,100,'ulw',plevs)

ulwdict = {'ctrl':ctrl_ulw,'mdd_p2f':mdd_p2f_ulw,'dd_p2f':dd_p2f_ulw}
with open(f"./store/haiyansen_memb07_ulw.pkl", "wb") as f:
    pickle.dump(ulwdict, f)

ctrl_mlw= proc_onevar_loop(ctrl_files,track_memb07,100,'mlw',plevs)
dd_p2f_mlw= proc_onevar_loop(dd_p2f_files,track_memb07,100,'mlw',plevs)
mdd_p2f_mlw= proc_onevar_loop(mdd_p2f_files,track_memb07,100,'mlw',plevs)

mlwdict = {'ctrl':ctrl_mlw,'mdd_p2f':mdd_p2f_mlw,'dd_p2f':dd_p2f_mlw}
with open(f"./store/haiyansen_memb07_mlw.pkl", "wb") as f:
    pickle.dump(mlwdict, f)

100%|██████████| 29/29 [02:02<00:00,  4.22s/it]


In [31]:
track_memb03 = xr.open_dataset('/glade/work/ihtam/storage/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/ctrl/wrfout_d02_2013-11-0*"))[:29]
miaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
aiaxi2x_files = sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/add_2idealaxi/wrfout_d02_2013-11-0*"))[:29]
mdd_p2f_files =sorted(glob.glob("/glade/campaign/univ/uokl0049/haiyan_pattern/minus_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]
dd_p2f_files = sorted(glob.glob("/glade/derecho/scratch/ihtam/haiyan/add_2DDp_pareto2/wrfout_d02_2013-11-0*"))[:29]

expname = ['ctrl','miaxi2x','aiaxi2x','mdd_p2f','dd_p2f']
files = [ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files]
del ctrl_files,miaxi2x_files,aiaxi2x_files,mdd_p2f_files,dd_p2f_files
gc.collect()
ulwdict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'ulw',plevs) for i in range(5)}
mlwdict_memb03 = {expname[i]:proc_onevar_loop(files[i],track_memb03,100,'mlw',plevs) for i in range(5)}

with open(f"./store/haiyansen_memb03_ulw.pkl", "wb") as f:
    pickle.dump(ulwdict_memb03, f)

with open(f"./store/haiyansen_memb03_mlw.pkl", "wb") as f:
    pickle.dump(mlwdict_memb03, f)

100%|██████████| 29/29 [01:46<00:00,  3.69s/it]


In [32]:
oldensm_ulw = {}
oldensm_mlw = {}
for i in ['01','02','03','04','05','06','07','08','09','10']:
    track_memb03 = xr.open_dataset(f'/glade/derecho/scratch/ihtam/temp/memb{i}/track_avor_850-600.nc')
    ctrl_files = sorted(glob.glob(f"/glade/derecho/scratch/ihtam/temp/memb{i}/wrfout_d02_2013-11-0*"))[1:30]
    oldensm_ulw[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'ulw',plevs)
    oldensm_mlw[f'memb{i}'] = proc_onevar_loop(ctrl_files,track_memb03,100,'mlw',plevs)
    
with open(f"./store/haiyan_oldensm_ulw.pkl", "wb") as f:
    pickle.dump(oldensm_ulw, f)
with open(f"./store/haiyan_oldensm_mlw.pkl", "wb") as f:
    pickle.dump(oldensm_mlw, f)

 62%|██████▏   | 18/29 [01:17<00:47,  4.29s/it]/glade/derecho/scratch/ihtam/tmp/ipykernel_110778/3775181567.py:44: RuntimeWarning: Mean of empty slice.
  varouts.append(float(ulw[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
 62%|██████▏   | 18/29 [01:15<00:45,  4.16s/it]/glade/derecho/scratch/ihtam/tmp/ipykernel_110778/3775181567.py:47: RuntimeWarning: Mean of empty slice.
  varouts.append(float(mlw[int(iy)-dr:int(iy)+dr,int(ix)-dr:int(ix)+dr].mean()))
/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
 69%|██████▉   | 20/29 [01:29<00:40,  4.46s/it]/glade/work/ihtam/miniconda3/envs/myenv/lib/python3.8/site-packages/wrf/latlonutils.py:434: RuntimeWarning: invalid v